In [26]:
# Basic Imports
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_ollama import ChatOllama
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables.history import RunnableWithMessageHistory

# ✅ Task 3.1.2: Naya Import - SQL wali Memory Class
from langchain_community.chat_message_histories import SQLChatMessageHistory

# --- Setup same rahega ---
prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful assistant."),
    MessagesPlaceholder(variable_name="chat_history"),
    ("human", "{input}")
])

model = ChatOllama(model="mistral:7b")
chain = prompt | model | StrOutputParser()

# ---------------------------------------------------------
# 🔥 THE COMBO TASK 3.1 & 3.2: RAM Store Hatao, SQL Lagao
# ---------------------------------------------------------
def get_session_history(session_id: str):
    # Ab dictionary check nahi hogi, seedha database return hoga
    return SQLChatMessageHistory(
        session_id=session_id,
        # ✅ Task 3.2.1: Connection String (relative path ke liye 3 slashes)
        connection_string="sqlite:///my_local_memory.db" 
    )

# Wrapper (Dependency Injection - Chain ko farq nahi padta memory kahan se aa rahi hai)
memory_chain = RunnableWithMessageHistory(
    chain,
    get_session_history,
    input_messages_key="input",
    history_messages_key="chat_history"
)

# --- Execution Test ---
print("Test 1 Running...")
resp1 = memory_chain.invoke(
    {"input": "Mera naam Karthik hai aur main Patna se hoon."},
    config={"configurable": {"session_id": "user_123"}}
)
print("AI:", resp1)

print("\nTest 2 Running...")
resp2 = memory_chain.invoke(
    {"input": "Main kahan se hoon?"},
    config={"configurable": {"session_id": "user_123"}}
)
print("AI:", resp2)

Test 1 Running...
AI:  Namaskar Karthikji! Aap Patna se ho sakte hain, main aapko bahut vishwasan utna madad karne ki jise meri avastha hai. Agar koi sawal hai ya pooja karna hai toh main apne sabse acchi tarah se usko jawaab denge.

(Hello Karthik! You are from Patna, I am here to help you as best as I can. If you have any questions or need something done, I will do my best to answer them.)

Test 2 Running...
AI:  Main internet se hoonga (I am from the internet). Lekin aapko jyada samajhna chahte ho toh main Patna mein rehta hoon, lekin internet se bhi dosti kar raha hoon.

(I'm from the internet but if you want more context, I am based in Patna but I also have a connection with it through the internet.)
